In [73]:
#!/usr/bin/env python3
"""
example_strategy.py

Direct translation of example_script.R:
- Maintain a lookback-window of closes (lagged_price)
- Compute 10-day momentum sign(close_t - close_{t-10})
- Target position per symbol = (wealth / num_symbols) * momentum_sign
- Trades = target_position - current_position

This matches walk_forward.py’s interface:
  initialise_state(df_train) -> state
  trading_algorithm(new_data, state) -> (trades, new_state)

Trades are in "wealth units" (dollar allocation), same as the R version.
"""

'\nexample_strategy.py\n\nDirect translation of example_script.R:\n- Maintain a lookback-window of closes (lagged_price)\n- Compute 10-day momentum sign(close_t - close_{t-10})\n- Target position per symbol = (wealth / num_symbols) * momentum_sign\n- Trades = target_position - current_position\n\nThis matches walk_forward.py’s interface:\n  initialise_state(df_train) -> state\n  trading_algorithm(new_data, state) -> (trades, new_state)\n\nTrades are in "wealth units" (dollar allocation), same as the R version.\n'

In [74]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Tuple
import numpy as np
import pandas as pd
from itertools import combinations
from statsmodels.tsa.stattools import coint, adfuller
import matplotlib.pyplot as plt

In [75]:
# Import data
df = pd.read_csv("df_train.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["date", "symbol"]).reset_index(drop=True)

# Walk-forward-style split
split_date = pd.Timestamp("2012-01-01")
train_df = df[df["date"] < split_date].copy()
test_df  = df[df["date"] >= split_date].copy()

print("Full data:", df.shape)
print("Train data:", train_df.shape)
print("Test data:", test_df.shape)

# Use TRAIN data for all pair detection / diagnostics
close_wide = train_df.pivot(index="date", columns="symbol", values="close").sort_index()
volume_wide = train_df.pivot(index="date", columns="symbol", values="volume").sort_index()

# Daily close-to-close returns on TRAIN only
ret_wide = close_wide.pct_change()

print("Train close matrix shape:", close_wide.shape)
print("Train return matrix shape:", ret_wide.shape)

Full data: (100600, 7)
Train data: (50400, 7)
Test data: (50200, 7)
Train close matrix shape: (504, 100)
Train return matrix shape: (504, 100)


In [76]:
# Highest pairwise return correlations
corr_matrix = ret_wide.corr()

pairs = []
symbols = corr_matrix.columns.tolist()

for i, j in combinations(symbols, 2):
    corr_val = corr_matrix.loc[i, j]
    pairs.append((i, j, corr_val))

pairs_df = pd.DataFrame(pairs, columns=["symbol_1", "symbol_2", "return_corr"])
pairs_df = pairs_df.sort_values("return_corr", ascending=False).reset_index(drop=True)

print("Top 20 most correlated pairs:")
print(pairs_df.head(20))

Top 20 most correlated pairs:
   symbol_1 symbol_2  return_corr
0       MQV      WQX     0.989399
1       FBR     GZLT     0.961714
2      CDRX     TLWM     0.957040
3       DLT      QVG     0.946318
4      GTPX      MNG     0.931964
5      BZQM      LZT     0.929981
6      GTFX      JFK     0.916764
7       BZK     NVDT     0.914689
8       MQV     SYDR     0.908696
9       MQV      WZB     0.906741
10     SYDR      WQX     0.905038
11      HLC      NQY     0.904013
12     RWXT     VKNT     0.901804
13     KZMB     NVDT     0.893453
14     BZQM      YPN     0.888328
15      WQX      WZB     0.887512
16      FJX      XTG     0.885248
17     JBDX     RFXP     0.884974
18     ACTS      CDM     0.883092
19     AMWD      JFK     0.878407


In [77]:
def estimate_hedge_ratio(log_y, log_x):
    """
    Regress log_y on log_x:
        log_y = alpha + beta * log_x + error
    Return alpha, beta, spread
    """
    X = np.column_stack([np.ones(len(log_x)), log_x.values])
    y = log_y.values
    alpha, beta = np.linalg.lstsq(X, y, rcond=None)[0]
    spread = log_y - (alpha + beta * log_x)
    return alpha, beta, spread


def estimate_half_life(spread):
    spread = spread.dropna()
    if len(spread) < 30:
        return np.nan

    lagged = spread.shift(1)
    current = spread

    reg_df = pd.concat([lagged, current], axis=1).dropna()
    reg_df.columns = ["lagged", "current"]

    X = np.column_stack([np.ones(len(reg_df)), reg_df["lagged"].values])
    y = reg_df["current"].values

    coef = np.linalg.lstsq(X, y, rcond=None)[0]
    rho = coef[1]

    if not np.isfinite(rho) or abs(rho) >= 1 or rho <= 0:
        return np.nan

    return np.log(0.5) / np.log(abs(rho))

In [78]:
def pair_cointegration_diagnostics(price_1, price_2, name_1="A", name_2="B"):
    pair_df = pd.concat([price_1, price_2], axis=1).dropna()
    pair_df.columns = [name_1, name_2]

    if len(pair_df) < 100:
        return None

    # log prices
    log_1 = np.log(pair_df[name_1])
    log_2 = np.log(pair_df[name_2])

    # return correlation
    ret_corr = pair_df.pct_change().corr().iloc[0, 1]

    # first-stage regression includes an intercept:
    # log_1 = alpha + beta * log_2 + residual
    alpha, beta, spread = estimate_hedge_ratio(log_1, log_2)

    # Engle-Granger ADF test on fitted residuals.
    # We keep the test statistic, but make the decision using lecture-style
    # critical values for: intercept = yes, trend = no.
    try:
        coint_stat, coint_pvalue, crit_vals = coint(log_1, log_2, trend="c")
    except Exception:
        coint_stat, coint_pvalue = np.nan, np.nan
        crit_vals = [np.nan, np.nan, np.nan]

    # EG critical values for intercept = yes, trend = no at 5% level
    eg_cv_5pct = -3.37 
    eg_sig_5pct = np.isfinite(coint_stat) and (coint_stat < eg_cv_5pct)

    # Keep spread ADF as descriptive extra information only
    try:
        adf_stat, adf_pvalue, *_ = adfuller(spread.dropna())
    except Exception:
        adf_stat, adf_pvalue = np.nan, np.nan

    half_life = estimate_half_life(spread)

    return {
        "symbol_1": name_1,
        "symbol_2": name_2,
        "return_corr": ret_corr,
        "coint_stat": coint_stat,
        "coint_pvalue": coint_pvalue,
        "eg_cv_5pct": eg_cv_5pct,
        "eg_sig_5pct": eg_sig_5pct,
        "adf_stat": adf_stat,
        "adf_pvalue": adf_pvalue,
        "hedge_ratio_beta": beta,
        "spread_mean": spread.mean(),
        "spread_std": spread.std(),
        "half_life": half_life
    }

In [79]:
top_n = 100
candidate_pairs = pairs_df.head(top_n)

results = []
for _, row in candidate_pairs.iterrows():
    s1 = row["symbol_1"]
    s2 = row["symbol_2"]

    diag = pair_cointegration_diagnostics(close_wide[s1], close_wide[s2], s1, s2)
    if diag is not None:
        results.append(diag)

cointegration_df = pd.DataFrame(results)

# Rank by most negative EG statistic first
cointegration_df = cointegration_df.sort_values(
    by=["coint_stat", "return_corr"],
    ascending=[True, False]
).reset_index(drop=True)

print("Cointegration diagnostics:")
print(cointegration_df[[
    "symbol_1", "symbol_2", "return_corr",
    "coint_stat", "eg_cv_5pct", "eg_sig_5pct", 
    "half_life", "hedge_ratio_beta"
]].head(20))

Cointegration diagnostics:
   symbol_1 symbol_2  return_corr  coint_stat  eg_cv_5pct  eg_sig_5pct  \
0      SYDR      WQX     0.905038   -4.152961       -3.37         True   
1      JBDX     JZKP     0.811899   -3.788040       -3.37         True   
2       CDM      HJC     0.824002   -3.536457       -3.37         True   
3       KRV     WYLR     0.843135   -3.425598       -3.37         True   
4       HJC      PWL     0.828210   -3.347356       -3.37        False   
5       BBY      HJC     0.795274   -3.343520       -3.37        False   
6      QVNR     WMGX     0.799985   -3.315783       -3.37        False   
7       VXT     WMGX     0.795876   -3.211225       -3.37        False   
8      RWXT     VKNT     0.901804   -3.127606       -3.37        False   
9       LZT      YPN     0.860204   -3.103928       -3.37        False   
10      FJX      XTG     0.885248   -3.062345       -3.37        False   
11      FJX      RYX     0.797574   -3.020009       -3.37        False   
12      JQR

In [80]:
# Strict usable pairs: EG significant at 5%
# intercept = yes, trend = no, critical value=-3.37
usable_pairs = cointegration_df[
    (cointegration_df["return_corr"] > 0.75) &
    (cointegration_df["eg_sig_5pct"] == True) &
    (cointegration_df["half_life"].notna()) &
    (cointegration_df["half_life"] > 1) &
    (cointegration_df["half_life"] < 60)
].copy()

strict_pairs_df = usable_pairs.sort_values(
    by=["coint_stat", "return_corr"],
    ascending=[True, False]
).reset_index(drop=True)

print("Usable cointegrated pairs (EG 5% critical-value rule):")
print(strict_pairs_df[[
    "symbol_1", "symbol_2", "return_corr",
    "coint_stat", "eg_cv_5pct", "eg_sig_5pct",
    "half_life", "hedge_ratio_beta"
]].head(20))

Usable cointegrated pairs (EG 5% critical-value rule):
  symbol_1 symbol_2  return_corr  coint_stat  eg_cv_5pct  eg_sig_5pct  \
0     SYDR      WQX     0.905038   -4.152961       -3.37         True   
1     JBDX     JZKP     0.811899   -3.788040       -3.37         True   
2      CDM      HJC     0.824002   -3.536457       -3.37         True   
3      KRV     WYLR     0.843135   -3.425598       -3.37         True   

   half_life  hedge_ratio_beta  
0  12.085395          0.893523  
1   9.892167          0.861499  
2  16.009490          1.388441  
3  15.186761          1.481454  


In [81]:
# I(1) test
# Wide close-price matrix
close_wide = train_df.pivot(index="date", columns="symbol", values="close").sort_index()

# Use the presplit usable pairs, not the old hardcoded pairs
pairs_to_check = list(
    usable_pairs[["symbol_1", "symbol_2"]].head(4).itertuples(index=False, name=None)
)

symbols_to_check = sorted(set(
    [x for pair in pairs_to_check for x in pair]
))

print("Pairs being checked:", pairs_to_check)
print("Symbols being checked:", symbols_to_check)

def safe_adf(x, name="series"):
    x = pd.Series(x).dropna()
    try:
        stat, pval, usedlag, nobs, crit, icbest = adfuller(x, autolag="AIC")
        return {
            "series": name,
            "adf_stat": stat,
            "pvalue": pval,
            "lags": usedlag,
            "nobs": nobs
        }
    except Exception as e:
        return {
            "series": name,
            "adf_stat": np.nan,
            "pvalue": np.nan,
            "lags": np.nan,
            "nobs": np.nan,
            "error": str(e)
        }

results = []

for sym in symbols_to_check:
    log_price = np.log(close_wide[sym]).dropna()
    diff_log_price = log_price.diff().dropna()

    results.append(safe_adf(log_price, name=f"{sym}_log_price"))
    results.append(safe_adf(diff_log_price, name=f"{sym}_diff_log_price"))

results_df = pd.DataFrame(results)
print(results_df)

Pairs being checked: [('SYDR', 'WQX'), ('JBDX', 'JZKP'), ('CDM', 'HJC'), ('KRV', 'WYLR')]
Symbols being checked: ['CDM', 'HJC', 'JBDX', 'JZKP', 'KRV', 'SYDR', 'WQX', 'WYLR']
                 series   adf_stat        pvalue  lags  nobs
0         CDM_log_price  -1.488209  5.393359e-01     3   500
1    CDM_diff_log_price -15.434617  2.918133e-28     2   500
2         HJC_log_price  -2.422151  1.355819e-01     2   501
3    HJC_diff_log_price -18.133143  2.502794e-30     1   501
4        JBDX_log_price  -2.448146  1.286059e-01    13   490
5   JBDX_diff_log_price  -5.256945  6.719635e-06    12   490
6        JZKP_log_price  -2.856977  5.058466e-02    13   490
7   JZKP_diff_log_price -17.737785  3.412674e-30     1   501
8         KRV_log_price  -2.299901  1.720000e-01     2   501
9    KRV_diff_log_price -17.259853  5.950775e-30     1   501
10       SYDR_log_price  -2.468971  1.232078e-01     0   503
11  SYDR_diff_log_price  -9.131549  3.012559e-15     7   495
12        WQX_log_price  -2.35994

In [82]:
# Display I(1)s
i1_symbols = []

for sym in symbols_to_check:
    p_level = results_df.loc[
        results_df["series"] == f"{sym}_log_price", "pvalue"
    ].iloc[0]

    p_diff = results_df.loc[
        results_df["series"] == f"{sym}_diff_log_price", "pvalue"
    ].iloc[0]

    if (p_level > 0.05) and (p_diff < 0.05):
        i1_symbols.append(sym)

print("Plausibly I(1) symbols:")
print(i1_symbols)

Plausibly I(1) symbols:
['CDM', 'HJC', 'JBDX', 'JZKP', 'KRV', 'SYDR', 'WQX', 'WYLR']


In [83]:
# I(1) pairs
i1_set = set(i1_symbols)

strict_i1_pairs = [
    pair for pair in strict_pairs
    if pair[0] in i1_set and pair[1] in i1_set
]

print("Strict pairs with both symbols plausibly I(1):")
print(strict_i1_pairs)

Strict pairs with both symbols plausibly I(1):
[('SYDR', 'WQX'), ('JBDX', 'JZKP'), ('CDM', 'HJC'), ('KRV', 'WYLR')]


In [84]:
@dataclass
class State:
    symbols: list[str]            # fixed symbol order
    lagged_price: np.ndarray      # shape (LOOKBACK, NUM_SYMBOLS), row 0 = most recent
    wealth: float                 # internal wealth tracker (matches R example logic)
    positions: np.ndarray         # current allocations, shape (NUM_SYMBOLS,)

In [85]:
# Pair trading strategy state initialization
def initialise_state(data):
    data = data.copy()
    data["date"] = pd.to_datetime(data["date"])
    data = data.sort_values(["date", "symbol"]).reset_index(drop=True)

    # Selected usable pairs detected from the TRAIN split only
    # (2010-2011 for the 2012-2013 test)
    selected_pairs = [
    # strict 5% pairs
    ("SYDR", "WQX", 0.893523),
    ("JBDX", "JZKP", 0.861499),
    ("CDM", "HJC", 1.388441),
    ("KRV", "WYLR", 1.481454),

    # borderline 10% pairs that also passed the I(1) sanity check
    #("HJC", "PWL", 0.529872),
    #("BBY", "HJC", 0.980445),
    #("QVNR", "WMGX", 1.240569),
    #("VXT", "WMGX", 1.310206),
    #("RWXT", "VKNT", 0.901378),
    #("LZT", "YPN", 0.795457),
    ]

    symbols = sorted(data["symbol"].unique().tolist())

    # Keep full close-price history by symbol
    close_hist = {
        sym: data.loc[data["symbol"] == sym, ["date", "close"]]
            .set_index("date")["close"]
            .sort_index()
            .copy()
        for sym in symbols
    }

    state = {
        "symbols": symbols,
        "selected_pairs": selected_pairs,
        "close_hist": close_hist,
        "current_weights": {sym: 0.0 for sym in symbols},

        # Trading rule parameters
        "lookback": 100,
        "entry_z": 1.5,
        "exit_z": 1.0,
        "pair_gross": 0.20,

        # +1 = long spread, -1 = short spread, 0 = flat
        "pair_position": {f"{y}_{x}": 0 for y, x, _ in selected_pairs}
    }

    return state

In [86]:
# Pair trading algorithm
def trading_algorithm(new_data, state):
    new_data = new_data.copy()
    new_data["date"] = pd.to_datetime(new_data["date"])
    new_data = new_data.sort_values("symbol").reset_index(drop=True)

    current_date = new_data["date"].iloc[0]

    # Update close history with today's close
    price_today = {}
    for _, row in new_data.iterrows():
        sym = row["symbol"]
        px = float(row["close"])
        price_today[sym] = px
        state["close_hist"][sym].loc[current_date] = px

    desired_weights = {sym: 0.0 for sym in state["symbols"]}

    lookback = state["lookback"]
    entry_z = state["entry_z"]
    exit_z = state["exit_z"]
    pair_gross = state["pair_gross"]

    for y_sym, x_sym, beta in state["selected_pairs"]:
        pair_key = f"{y_sym}_{x_sym}"

        y_hist = state["close_hist"][y_sym].sort_index().dropna()
        x_hist = state["close_hist"][x_sym].sort_index().dropna()

        pair_df = pd.concat([y_hist, x_hist], axis=1, join="inner")
        pair_df.columns = ["y", "x"]

        # Need enough history
        if len(pair_df) < lookback:
            continue

        # Log-price spread: U_t = log(Y_t) - beta * log(X_t)
        spread = np.log(pair_df["y"]) - beta * np.log(pair_df["x"])

        roll_mean = spread.rolling(lookback).mean()
        roll_std = spread.rolling(lookback).std()

        spread_t = spread.iloc[-1]
        mu_t = roll_mean.iloc[-1]
        sd_t = roll_std.iloc[-1]

        if pd.isna(mu_t) or pd.isna(sd_t) or sd_t <= 1e-12:
            continue

        z_t = (spread_t - mu_t) / sd_t

        # Price-level hedge ratio from the note:
        # dX2 = beta * X2/X1 * dX1
        # so hedge ratio in units is beta * Y/X if spread = log(Y) - beta log(X)
        y_px = price_today[y_sym]
        x_px = price_today[x_sym]
        hedge_units = beta * (y_px / x_px)

        # Convert to dollar weights.
        # We normalize so total gross pair exposure is approximately pair_gross.
        # One "bundle" is:
        #   +1 unit Y and -hedge_units unit X   (long spread)
        # or
        #   -1 unit Y and +hedge_units unit X   (short spread)
        gross_bundle = 1.0 + abs(hedge_units)
        if gross_bundle <= 1e-12:
            continue

        scale = pair_gross / gross_bundle

        current_pos = state["pair_position"][pair_key]
        target_pos = current_pos

        # Signal logic from the note:
        # spread high => short spread
        # spread low  => long spread
        # close when spread reverts near mean
        if current_pos == 0:
            if z_t > entry_z:
                target_pos = -1   # short spread
            elif z_t < -entry_z:
                target_pos = 1    # long spread
        else:
            if abs(z_t) < exit_z:
                target_pos = 0

        state["pair_position"][pair_key] = target_pos

        if target_pos == 1:
            # Long spread: +Y, -hedged X
            desired_weights[y_sym] += scale
            desired_weights[x_sym] += -scale * hedge_units

        elif target_pos == -1:
            # Short spread: -Y, +hedged X
            desired_weights[y_sym] += -scale
            desired_weights[x_sym] += scale * hedge_units

    # Optional safety cap on total gross exposure
    gross = sum(abs(w) for w in desired_weights.values())
    max_gross = 0.25
    if gross > max_gross and gross > 1e-12:
        shrink = max_gross / gross
        desired_weights = {k: v * shrink for k, v in desired_weights.items()}

    # Trades = desired weights - current weights
    trades = {
        sym: desired_weights[sym] - state["current_weights"].get(sym, 0.0)
        for sym in state["symbols"]
    }

    state["current_weights"] = desired_weights

    # Return as named vector-like object
    trades = pd.Series(trades)
    return trades, state